# GroupChatManager and Speaker Selection

In **AutoGen 0.2**, `GroupChatManager` orchestrated multi-agent chats by choosing the next speaker from a fixed participant list. In **AutoGen 0.4+ AgentChat**, that responsibility moves to **team presets** — especially **`SelectorGroupChat`**, which uses an LLM (or your own function) to pick the next speaker from **participant names and descriptions**.

This notebook covers **dynamic speaker selection**, **participant descriptions**, a **round-robin vs selector** comparison, a **legacy mapping table**, **custom `selector_prompt`**, and a **Notes API team** with dynamic routing.

**Prerequisites:** **02. Setup and the AgentChat API**, **08. GroupChat and Multi-Agent Teams**, **13. Termination Conditions and Guardrails**.

**Cross-references:** **10. Custom Agent Roles** (system prompts), **11. Sequential and Hierarchical Workflows** (pipeline vs conversation), **LangGraph/11** (supervisor routing).

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
from typing import Sequence

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import AgentEvent, ChatMessage, TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

---

## 1. Legacy `GroupChatManager` → AgentChat Teams

AutoGen 0.4 is a rewrite. Group chat orchestration is no longer a separate manager class — it is built into **team** types:

| AutoGen 0.2 (legacy) | AutoGen 0.4+ AgentChat | Speaker selection |
|----------------------|------------------------|-------------------|
| `GroupChat` + `GroupChatManager` | `RoundRobinGroupChat` | Fixed rotation |
| `GroupChat` + `speaker_selection_method="auto"` | `SelectorGroupChat` | LLM picks next speaker |
| `GroupChat` + custom speaker function | `SelectorGroupChat(selector_func=...)` | Your Python function |
| `GroupChat` + agent handoffs | `Swarm` | `HandoffMessage` target (**10**) |
| Magentic-One orchestrator | `MagenticOneGroupChat` | Built-in lead orchestrator (**11**) |

```text
v0.2:  User → GroupChatManager.select_speaker() → Agent → broadcast
v0.4:  User → Team.run_stream() → select_speaker (LLM/func) → Agent → broadcast
```

All participants still **share one message thread** — agents broadcast to everyone. Selection only decides **who speaks next**.

---

## 2. How `SelectorGroupChat` Chooses Speakers

When you call `await team.run_stream(task=...)`:

1. The team reads **conversation history** plus each agent's **`name`** and **`description`**.
2. A **selector model** (or `selector_func`) returns the next participant name.
3. That agent generates a response (tools optional); the message is **broadcast** to all members.
4. **Termination** is checked (**13**); repeat until done.

By default **`allow_repeated_speaker=False`** — the same agent cannot speak twice in a row unless it is the only option. Set `allow_repeated_speaker=True` for back-to-back turns.

---

## 3. Notes API Corpus and Teaching Tools

Same corpus as **LangGraph/05–11** — keyword search and endpoint stubs for multi-agent demos:

In [ ]:
DOC_CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


def search_docs(query: str) -> str:
    """Keyword search over Notes API documentation chunks."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


def describe_endpoint(method: str, path: str) -> str:
    """Describe a Notes API HTTP endpoint (teaching stub)."""
    catalog = {
        ("GET", "/notes"): "List all notes for the authenticated user.",
        ("POST", "/notes"): "Create a note; body requires title and content.",
        ("PUT", "/notes/{id}"): "Update an existing note by id.",
        ("DELETE", "/notes/{id}"): "Delete a note by id.",
    }
    return catalog.get((method.upper(), path), f"Unknown endpoint {method} {path}")

In [ ]:
def make_model_client():
    return OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

model_client = make_model_client()

In [ ]:
text_termination = TextMentionTermination("TERMINATE")
max_msg_termination = MaxMessageTermination(max_messages=20)
default_termination = text_termination | max_msg_termination

---

## 4. Participant Descriptions Drive Selection

The selector LLM uses **`description`** (not only `system_message`) to route work. Write descriptions as **routing hints**:

| Agent | `name` | `description` purpose |
|-------|--------|------------------------|
| Planner | `notes_planner` | First speaker; breaks tasks into steps |
| Docs | `docs_specialist` | Searches documentation chunks |
| API | `api_specialist` | Describes HTTP endpoints |

**Tip:** Include *when* the agent should speak ("use after planner assigns a docs task").

In [ ]:
notes_planner = AssistantAgent(
    "notes_planner",
    description="Planning agent for Notes API tasks. Should speak first on new tasks and last to summarize.",
    model_client=model_client,
    system_message=(
        "You are a planning agent for the FastAPI Notes API.\n"
        "Break tasks into steps and assign docs_specialist or api_specialist.\n"
        "Do not call tools yourself. End with TERMINATE when done."
    ),
)

docs_specialist = AssistantAgent(
    "docs_specialist",
    description="Documentation specialist. Searches Notes API doc chunks with search_docs.",
    model_client=model_client,
    tools=[search_docs],
    system_message="Search documentation with search_docs. Cite chunk ids like [c2]. Be concise.",
)

api_specialist = AssistantAgent(
    "api_specialist",
    description="API specialist. Describes Notes API HTTP endpoints with describe_endpoint.",
    model_client=model_client,
    tools=[describe_endpoint],
    system_message="Use describe_endpoint for route questions. Mention method and path.",
)

---

## 5. `RoundRobinGroupChat` — Fixed Turn Order

Round-robin ignores task context: agents speak in list order. Useful for **pipelines** where order is always A → B → C (**11**).

In [ ]:
round_robin_team = RoundRobinGroupChat(
    [notes_planner, docs_specialist, api_specialist],
    termination_condition=default_termination,
)

ROUND_ROBIN_TASK = (
    "Step 1: docs_specialist search Alembic migrations. "
    "Step 2: api_specialist describe GET /notes. Planner summarizes."
)

# await Console(round_robin_team.run_stream(task=ROUND_ROBIN_TASK))
print("RoundRobinGroupChat ready:", [a.name for a in round_robin_team._participants])

**Observation:** Planner may speak again before specialists finish — round-robin does not wait for "assignment complete." That is why selector teams are preferred for **dynamic** collaboration.

---

## 6. `SelectorGroupChat` — LLM Speaker Selection

The selector model reads history + descriptions and picks the best next speaker:

In [ ]:
selector_team = SelectorGroupChat(
    [notes_planner, docs_specialist, api_specialist],
    model_client=model_client,
    termination_condition=default_termination,
)

SELECTOR_TASK = (
    "For the Notes API: (1) find how JWT auth works in docs, "
    "(2) describe the POST /notes endpoint, then summarize."
)

# await Console(selector_team.run_stream(task=SELECTOR_TASK))
print("SelectorGroupChat ready with selector model:", model_client.model)

---

## 7. Round-Robin vs Selector — Comparison

| Dimension | `RoundRobinGroupChat` | `SelectorGroupChat` |
|-----------|----------------------|---------------------|
| Next speaker | List index rotation | LLM or `selector_func` |
| Uses `description` | No | Yes |
| Extra LLM call per turn | No | Yes (selection step) |
| Predictable order | ✅ | ❌ (context-driven) |
| Skips irrelevant agents | ❌ | ✅ |
| Best for | Fixed pipelines, debates | Dynamic task routing |

```text
Round-robin:  planner → docs → api → planner → docs → ...
Selector:     planner → docs → docs → api → planner → TERMINATE
```

---

## 8. Custom `selector_prompt`

Override the default role-play prompt. Placeholders **`{roles}`**, **`{participants}`**, and **`{history}`** are required:

In [ ]:
NOTES_SELECTOR_PROMPT = """You coordinate a Notes API support team.

Available roles:
{roles}

Conversation so far:
{history}

Rules:
- Pick notes_planner first on a new user task.
- Pick docs_specialist for documentation or Alembic/JWT/pytest questions.
- Pick api_specialist for HTTP route or endpoint questions.
- After both specialists contribute, pick notes_planner to summarize.

Participants: {participants}
Reply with ONLY the agent name to speak next."""

custom_prompt_team = SelectorGroupChat(
    [notes_planner, docs_specialist, api_specialist],
    model_client=model_client,
    termination_condition=default_termination,
    selector_prompt=NOTES_SELECTOR_PROMPT,
)
print("Custom selector_prompt configured")

---

## 9. Custom `selector_func` — Deterministic Overrides

Return a **participant name** to force the next speaker. Return **`None`** to fall back to the default LLM selector (per AutoGen docs):

In [ ]:
SPECIALISTS = {"docs_specialist", "api_specialist"}


def notes_selector_func(messages: Sequence[AgentEvent | ChatMessage]) -> str | None:
    """After any specialist speaks, return to planner for checkpoint."""
    if not messages:
        return "notes_planner"
    last = messages[-1]
    source = getattr(last, "source", "")
    if source in SPECIALISTS:
        return "notes_planner"
    return None  # defer to LLM selector


func_team = SelectorGroupChat(
    [notes_planner, docs_specialist, api_specialist],
    model_client=model_client,
    termination_condition=default_termination,
    selector_func=notes_selector_func,
)
print("selector_func: planner follows every specialist turn")

**Pattern:** Combine **LLM flexibility** with **hard rules** — e.g., always return to planner after tool use, or force `api_specialist` when the user message contains `/notes`.

---

## 10. Inspect Speaker Order from `TaskResult`

After `run` / `run_stream`, walk `TaskResult.messages` and print `source` to audit routing:

In [ ]:
async def speaker_trace(team, task: str) -> list[str]:
    result = await team.run(task=task)
    order = [m.source for m in result.messages if hasattr(m, "source")]
    print("Speaker order:", " → ".join(order))
    return order


# order = await speaker_trace(selector_team, "Search docs for JWT, then describe DELETE /notes/{id}")
print("Use: await speaker_trace(selector_team, task)")

---

## 11. `build_notes_selector_team()` Factory

Package the Notes API selector team for reuse in **11** and **12**:

In [ ]:
def build_notes_selector_team(*, selector_func=None, selector_prompt=None):
    planner = AssistantAgent(
        "notes_planner",
        description="Planning agent. Speaks first and last; assigns specialists.",
        model_client=model_client,
        system_message="Plan Notes API tasks, delegate, summarize, end with TERMINATE.",
    )
    docs = AssistantAgent(
        "docs_specialist",
        description="Documentation search via search_docs.",
        model_client=model_client,
        tools=[search_docs],
        system_message="Use search_docs; cite [c#] ids.",
    )
    api = AssistantAgent(
        "api_specialist",
        description="HTTP endpoint reference via describe_endpoint.",
        model_client=model_client,
        tools=[describe_endpoint],
        system_message="Use describe_endpoint for routes.",
    )
    kwargs = {}
    if selector_func is not None:
        kwargs["selector_func"] = selector_func
    if selector_prompt is not None:
        kwargs["selector_prompt"] = selector_prompt
    return SelectorGroupChat(
        [planner, docs, api],
        model_client=model_client,
        termination_condition=default_termination,
        **kwargs,
    )


team = build_notes_selector_team()
print("factory team:", type(team).__name__)

---

## 12. Async Streaming with `Console`

AgentChat teams are **async-first**. In notebooks, top-level `await` works; in scripts use `asyncio.run()`:

In [ ]:
async def run_notes_team_demo():
    team = build_notes_selector_team()
    task = "Find Alembic upgrade instructions and describe POST /notes."
    stream = team.run_stream(task=task)
    await Console(stream)


# await run_notes_team_demo()
print("Demo: await run_notes_team_demo()")

---

## 13. Reset vs Continue Context

- **`await team.reset()`** — clears team + participant message buffers (new conversation).
- **Second `run_stream` without reset** — continues from prior context (**12**).

Call `reset()` between unrelated user sessions on the same team instance.

In [ ]:
# await selector_team.reset()
print("Call await team.reset() between unrelated tasks")

---

## 14. Selection Cost and Guardrails

Each selector turn adds one **small LLM call** before the speaker agent runs. Mitigations:

- Use `RoundRobinGroupChat` when order is fixed (**11**).
- Use `selector_func` to skip the LLM for obvious routing.
- Cap turns with `MaxMessageTermination` (**13**).
- Keep `description` strings short and distinct.

---

## 15. Summary and Next Steps

| Concept | API |
|---------|-----|
| LLM picks next speaker | `SelectorGroupChat` |
| Fixed rotation | `RoundRobinGroupChat` |
| Routing metadata | `AssistantAgent(..., description=...)` |
| Custom selection prompt | `selector_prompt=` |
| Custom selection logic | `selector_func=` |
| Legacy manager | Replaced by team presets (table §1) |

**Next:** **10. Custom Agent Roles and System Messages** — craft planner/executor/reviewer prompts and preview `HandoffMessage` / `Swarm`.